# Game conditions: weather, start times and umpires

Builds one row per MLB game for 2022-2026 (all game types) with the conditions MLB reports for it: temperature, sky or roof condition, wind, scheduled and actual start, duration, delays, attendance and umpires. Jeffrey wrote the collection and extraction code.

It has two parts:

1. **Raw game feeds** downloads every game's live feed and box score from the MLB Stats API into one JSON Lines file, `data_files/game_score_raw_json.jsonl` (about 17 GB, about 14,600 games). The file is ignored by Git and shared through the team's Google folder.
2. **Game conditions table** streams that file one line at a time and saves `data/games/mlb_game_conditions.csv` (tracked, keyed by `game_pk`).

The saved CSV is enough for the simulator. To rebuild it from a copy of the raw file, put the file in `data_files/`, then run every cell except **Run the download**. That cell makes about 29,000 API requests; already-saved games are skipped.

**Timing:** MLB records these conditions at game time. They are observations, not pregame forecasts; don't treat them as information available before a game.

**Saved outputs** come from the run before the notebook moved to `notebooks/`, so one line shows the table's earlier `csv_files/` location.

# Raw game feeds

## Imports and configuration

In [1]:
import json
import time
from pathlib import Path

import requests

BASE_URL = "https://statsapi.mlb.com/api/v1"
GAME_URL = "https://statsapi.mlb.com/api/v1.1/game"

# Find the repository root so paths are the same whether Jupyter starts there or in notebooks/.
REPO_ROOT = next((folder for folder in [Path.cwd(), *Path.cwd().parents]
                  if (folder / "requirements.txt").is_file()
                  and (folder / "notebooks" / "weather_scrape.ipynb").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the repository root. Start Jupyter inside the repository.")

# Raw game feeds, about 17 GB. Git ignores data_files/; the file is shared through the team's Google folder.
RAW_JSON_DIR = REPO_ROOT / "data_files"
OUTPUT_FILE = RAW_JSON_DIR / "game_score_raw_json.jsonl"
FAILED_FILE = RAW_JSON_DIR / "mlb_failed_games.txt"

## Request helpers

- `get_json` retries a failed request up to five times, waiting 1, 2, 4 and 8 seconds.
- `get_games` requests each year's schedule for every game type (spring training, regular season, postseason, All-Star and exhibition) and keys it by `gamePk`. A postponed game appears on its original and its new date with the same `gamePk`; the later schedule record is kept.

In [1]:
session = requests.Session()


def get_json(url, params=None):
    """GET JSON from the MLB Stats API, retrying up to five times with waits of 1, 2, 4 and 8 seconds."""
    for attempt in range(5):
        try:
            response = session.get(url, params=params, timeout=60)
            response.raise_for_status()
            return response.json()
        except (requests.RequestException, ValueError) as error:
            if attempt == 4:
                raise error
            time.sleep(2 ** attempt)


def get_games(start_year, end_year):
    """Schedule records for every game type in each year, keyed by gamePk. A postponed game keeps its later record."""
    games = {}
    for year in range(start_year, end_year + 1):
        params = {"sportId": 1, "startDate": f"{year}-01-01", "endDate": f"{year}-12-31", "gameType": "S,R,F,D,L,W,A,E"}
        schedule_data = get_json(f"{BASE_URL}/schedule", params=params)
        for date in schedule_data.get("dates", []):
            for game in date.get("games", []):
                games[game["gamePk"]] = game
        print(f"Retrieved schedule for {year}")
    return games

## Download helpers

Each line of the raw file is one game: `game_pk`, its schedule record, the live feed (`/api/v1.1/game/{gamePk}/feed/live`) and the box score. A rerun reads the `game_pk` of every saved line and skips those games, so an interrupted download resumes. Games that fail are logged to `data_files/mlb_failed_games.txt`.

In [1]:
def get_completed_ids():
    """gamePks already saved in the raw file, so a rerun skips them."""
    completed = set()
    if OUTPUT_FILE.exists():
        with OUTPUT_FILE.open("r", encoding="utf-8") as file:
            for line in file:
                if line.strip():
                    record = json.loads(line)
                    completed.add(record["game_pk"])
    return completed


def get_raw_game(game_pk, schedule_record):
    """One raw-file record: the game's schedule record, live feed and box score."""
    feed_url = f"{GAME_URL}/{game_pk}/feed/live"
    box_url = f"{BASE_URL}/game/{game_pk}/boxscore"
    feed_data = get_json(feed_url)
    box_data = get_json(box_url)
    return {"game_pk": game_pk, "schedule": schedule_record, "feed": feed_data, "boxscore": box_data}


def scrape_games(start_year, end_year):
    """Append every game not yet saved to the raw file, one JSON line per game, and log games that fail."""
    OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
    games = get_games(start_year, end_year)
    completed = get_completed_ids()
    print(f"Total games found: {len(games)}")
    print(f"Previously downloaded: {len(completed)}")

    with OUTPUT_FILE.open("a", encoding="utf-8") as file:
        for index, (game_pk, schedule) in enumerate(games.items(), start=1):
            if game_pk in completed:
                continue
            try:
                record = get_raw_game(game_pk, schedule)
                json.dump(record, file)
                file.write("\n")
                file.flush()   # each finished game is on disk, so an interrupted run loses at most one game
                completed.add(game_pk)
            except (requests.RequestException, ValueError) as error:
                print(f"Failed game {game_pk}: {error}")
                with FAILED_FILE.open("a", encoding="utf-8") as failed:
                    failed.write(f"{game_pk}: {error}\n")
            if index % 100 == 0:
                print(f"Processed {index}/{len(games)} games")
            time.sleep(0.2)   # pause between games; requests run one at a time

    print("Scraping finished.")
    print(f"Total games saved: {len(completed)}")

## Run the download

Network requests happen only here. Skip this cell to work from an existing raw file.

In [1]:
scrape_games(2022, 2026)

Retrieved schedule for 2022
Retrieved schedule for 2023
Retrieved schedule for 2024
Retrieved schedule for 2025
Retrieved schedule for 2026
Total games found: 14595
Previously downloaded: 0
Processed 100/14595 games
Processed 200/14595 games
Processed 300/14595 games
Processed 400/14595 games
Processed 500/14595 games
Processed 600/14595 games
Processed 700/14595 games
Processed 800/14595 games
Processed 900/14595 games
Processed 1000/14595 games
Processed 1100/14595 games
Processed 1200/14595 games
Processed 1300/14595 games
Processed 1400/14595 games
Processed 1500/14595 games
Processed 1600/14595 games
Processed 1700/14595 games
Processed 1800/14595 games
Processed 1900/14595 games
Processed 2000/14595 games
Processed 2100/14595 games
Processed 2200/14595 games
Processed 2300/14595 games
Processed 2400/14595 games
Processed 2500/14595 games
Processed 2600/14595 games
Processed 2700/14595 games
Processed 2800/14595 games
Processed 2900/14595 games
Processed 3000/14595 games
Processed

# Game conditions table

## Imports and configuration

Game type codes come from the schedule. Umpire positions are listed under two naming styles because box scores use both.

In [23]:
import json
import re
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import pandas as pd

# REPO_ROOT and RAW_JSON_DIR come from the configuration cell under "Raw game feeds".
file_path = RAW_JSON_DIR / "game_score_raw_json.jsonl"
output_path = REPO_ROOT / "data" / "games" / "mlb_game_conditions.csv"

game_types = {"S": "Spring Training", "R": "Regular Season", "F": "Wild Card Series", "D": "Division Series",
              "L": "League Championship Series", "W": "World Series", "A": "All-Star Game", "E": "Exhibition"}

# Box scores name umpire positions in full or abbreviated
umpire_positions = {"Home Plate": "umpire_hp", "First Base": "umpire_1b", "Second Base": "umpire_2b",
                    "Third Base": "umpire_3b", "Left Field": "umpire_lf", "Right Field": "umpire_rf",
                    "HP": "umpire_hp", "1B": "umpire_1b", "2B": "umpire_2b", "3B": "umpire_3b",
                    "LF": "umpire_lf", "RF": "umpire_rf"}

columns = [
    "game_pk", "season", "game_type_code", "game_type", "game_status", "game_date", "venue",
    # conditions: degrees Fahrenheit, sky or roof condition, mph and direction
    "temperature_f", "weather_condition", "wind_speed_mph", "wind_direction",
    # times in US Eastern; *_minutes are minutes after midnight Eastern
    "scheduled_start_et", "scheduled_start_et_minutes", "first_pitch_et", "first_pitch_et_minutes", "first_pitch_source",
    "game_duration_minutes", "delay_duration_minutes", "attendance",
    "umpire_hp", "umpire_1b", "umpire_2b", "umpire_3b", "umpire_lf", "umpire_rf",
]

## Parsing helpers

- Temperatures are degrees Fahrenheit and wind speeds are mph, as MLB reports them.
- Wind text such as `7 mph, Out To CF` is split into speed and direction.
- Times are converted from UTC to US Eastern. The `*_minutes` columns are minutes after midnight Eastern.

In [23]:
def parse_integer(value):
    """First integer in a value such as "31,422" or "72 degrees", or None."""
    if value is None or isinstance(value, bool):
        return None
    match = re.search(r"-?\d[\d,]*", str(value))
    return int(match.group().replace(",", "")) if match else None


def parse_duration(value):
    """Minutes in an "h:mm" duration, or None."""
    if value is None:
        return None
    match = re.fullmatch(r"(\d+):(\d{2})", str(value).strip())
    if match:
        return int(match.group(1)) * 60 + int(match.group(2))
    return None


def parse_wind(value):
    """Speed and direction from wind text such as "7 mph, Out To CF"."""
    if not value:
        return None, None
    parts = str(value).split(",", 1)
    speed = parse_integer(parts[0])
    direction = parts[1].strip() if len(parts) > 1 else None
    return speed, direction


def parse_weather(value):
    """Temperature and condition from box-score weather text such as "72 degrees, Partly Cloudy"."""
    if not value:
        return None, None
    text = str(value).strip()
    temperature = parse_integer(text)
    parts = text.split(",", 1)
    if len(parts) > 1:
        condition = parts[1].strip()
    elif temperature is None:
        condition = text
    else:
        condition = None
    return temperature, condition


def convert_to_eastern(timestamp):
    """A UTC timestamp as an ISO time in US Eastern, plus minutes after midnight Eastern. None if it has no zone."""
    if not timestamp:
        return None, None
    try:
        date_time = datetime.fromisoformat(timestamp.replace("Z", "+00:00"))
        if date_time.tzinfo is None:
            return None, None
        eastern = date_time.astimezone(ZoneInfo("America/New_York"))
        minutes = eastern.hour * 60 + eastern.minute
        return eastern.isoformat(), minutes
    except (ValueError, TypeError):
        return None, None

## Extract one game

Fields come from the schedule record and the live feed's `gameData` first. Box score notes fill only what the feed left empty, and are the only source of delay length. `weather_condition` holds the sky condition, or `Roof Closed` / `Dome` when the game was played indoors.

In [23]:
def extract_game(record):
    """One row of the conditions table from one raw-file record. Box-score notes fill only fields the feed left empty."""
    schedule = record.get("schedule", {})
    feed = record.get("feed", {})
    boxscore = record.get("boxscore", {})
    game_data = feed.get("gameData", {})
    game_info = game_data.get("gameInfo", {})
    weather_data = game_data.get("weather", {})
    datetime_data = game_data.get("datetime", {})
    status = schedule.get("status", {})
    venue_data = schedule.get("venue", {})
    game_type_code = schedule.get("gameType")

    result = dict.fromkeys(columns)
    result["game_pk"] = record.get("game_pk")
    result["season"] = parse_integer(schedule.get("season"))
    result["game_type_code"] = game_type_code
    result["game_type"] = game_types.get(game_type_code, "Unknown")
    result["game_status"] = status.get("detailedState")
    result["game_date"] = schedule.get("officialDate")
    result["venue"] = venue_data.get("name")
    result["attendance"] = parse_integer(game_info.get("attendance"))
    result["game_duration_minutes"] = parse_integer(game_info.get("gameDurationMinutes"))
    result["temperature_f"] = parse_integer(weather_data.get("temp"))
    result["weather_condition"] = weather_data.get("condition")
    wind_speed, wind_direction = parse_wind(weather_data.get("wind"))
    result["wind_speed_mph"] = wind_speed
    result["wind_direction"] = wind_direction

    # Scheduled start and actual first pitch, both from UTC to Eastern
    scheduled_start, scheduled_minutes = convert_to_eastern(datetime_data.get("dateTime"))
    result["scheduled_start_et"] = scheduled_start
    result["scheduled_start_et_minutes"] = scheduled_minutes
    first_pitch, first_pitch_minutes = convert_to_eastern(game_info.get("firstPitch"))
    result["first_pitch_et"] = first_pitch
    result["first_pitch_et_minutes"] = first_pitch_minutes
    if first_pitch is not None:
        result["first_pitch_source"] = "gameInfo.firstPitch"

    for official in boxscore.get("officials", []):
        position = official.get("officialType")
        person = official.get("official", {})
        column = umpire_positions.get(position)
        if column:
            result[column] = person.get("fullName")

    # Box-score notes: fill gaps in the feed, and read delay length, which only the notes give
    for item in boxscore.get("info", []):
        label = str(item.get("label") or "").strip().rstrip(":").lower()
        value = item.get("value") or ""
        if label == "weather" and value:
            temperature, condition = parse_weather(value)
            if result["temperature_f"] is None:
                result["temperature_f"] = temperature
            if result["weather_condition"] is None:
                result["weather_condition"] = condition
        elif label == "wind" and value:
            speed, direction = parse_wind(value)
            if result["wind_speed_mph"] is None:
                result["wind_speed_mph"] = speed
            if result["wind_direction"] is None:
                result["wind_direction"] = direction
        elif label in ("att", "attendance"):
            if result["attendance"] is None:
                result["attendance"] = parse_integer(value)
        elif label in ("t", "time"):
            if result["game_duration_minutes"] is None:
                result["game_duration_minutes"] = parse_duration(value.split("(")[0].strip())
            delay_match = re.search(r"\((\d+:\d{2})\s+delay\)", value, flags=re.IGNORECASE)
            if delay_match:
                result["delay_duration_minutes"] = parse_duration(delay_match.group(1))
        elif label == "venue" and value:
            if result["venue"] is None:
                result["venue"] = value.rstrip(".")
    return result

## Read the raw file and save the table

The raw file is read one line at a time, so it is never loaded into memory whole. The summary below checks rows, failures, duplicate `game_pk` values and coverage by season, type and status.

In [23]:
# Stream the raw file one line at a time; it is too large to load whole
records = []
failed = 0
with file_path.open("r", encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        if not line.strip():
            continue
        try:
            record = json.loads(line)
            records.append(extract_game(record))
        except (ValueError, TypeError, KeyError, AttributeError) as error:
            failed += 1
            print(f"Failed line {line_number}: {error}")
        if line_number % 1000 == 0:
            print(f"Read {line_number} lines")

clean_df = pd.DataFrame(records, columns=columns)
# Nullable integers, so a missing temperature or attendance stays empty instead of turning the column into floats
integer_columns = ["game_pk", "season", "temperature_f", "wind_speed_mph", "scheduled_start_et_minutes",
                   "first_pitch_et_minutes", "game_duration_minutes", "delay_duration_minutes", "attendance"]
for column in integer_columns:
    clean_df[column] = pd.to_numeric(clean_df[column], errors="coerce").astype("Int64")

output_path.parent.mkdir(parents=True, exist_ok=True)
clean_df.to_csv(output_path, index=False)

# Rows, failures, duplicate game IDs, and coverage by season, type and status
print("Rows:", clean_df.shape[0])
print("Columns:", clean_df.shape[1])
print("Failed records:", failed)
print("Duplicate game IDs:", clean_df["game_pk"].duplicated().sum())
print("Saved to:", output_path.relative_to(REPO_ROOT))
print("\nGames by season:")
print(clean_df["season"].value_counts().sort_index())
print("\nGames by type:")
print(clean_df["game_type"].value_counts())
print("\nGames by status:")
print(clean_df["game_status"].value_counts())
print("\nNon-null values:")
print(clean_df.notna().sum())
display(clean_df.head(10))

Read 1000 lines
Read 2000 lines
Read 3000 lines
Read 4000 lines
Read 5000 lines
Read 6000 lines
Read 7000 lines
Read 8000 lines
Read 9000 lines
Read 10000 lines
Read 11000 lines
Read 12000 lines
Read 13000 lines
Read 14000 lines
Rows: 14595
Columns: 25
Failed records: 0
Duplicate game IDs: 0
Saved to: csv_files/mlb_game_conditions.csv

Games by season:
season
2022    2752
2023    2963
2024    2959
2025    2961
2026    2960
Name: count, dtype: Int64

Games by type:
game_type
Regular Season                12150
Spring Training                2142
Exhibition                       87
Division Series                  82
League Championship Series       55
Wild Card Series                 46
World Series                     28
All-Star Game                     5
Name: count, dtype: int64

Games by status:
game_status
Final              14480
Completed Early       58
Cancelled             57
Name: count, dtype: int64

Non-null values:
game_pk                       14595
season                

,game_pk,season,game_type_code,game_type,game_status,game_date,venue,temperature_f,weather_condition,wind_speed_mph,...,first_pitch_source,game_duration_minutes,delay_duration_minutes,attendance,umpire_hp,umpire_1b,umpire_2b,umpire_3b,umpire_lf,umpire_rf
0,663809,2022,S,Spring Training,Cancelled,2022-02-27,TD Ballpark,<NA>,NaN,<NA>,...,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
1,663808,2022,S,Spring Training,Cancelled,2022-03-01,TD Ballpark,<NA>,NaN,<NA>,...,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
2,663784,2022,S,Spring Training,Cancelled,2022-03-03,TD Ballpark,<NA>,NaN,<NA>,...,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
3,663779,2022,S,Spring Training,Cancelled,2022-03-04,TD Ballpark,<NA>,NaN,<NA>,...,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
4,663778,2022,S,Spring Training,Cancelled,2022-03-06,TD Ballpark,<NA>,NaN,<NA>,...,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
5,663802,2022,S,Spring Training,Cancelled,2022-03-09,TD Ballpark,<NA>,NaN,<NA>,...,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
6,663799,2022,S,Spring Training,Cancelled,2022-03-11,TD Ballpark,<NA>,NaN,<NA>,...,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
7,663777,2022,S,Spring Training,Cancelled,2022-03-13,TD Ballpark,<NA>,NaN,<NA>,...,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
8,663791,2022,S,Spring Training,Cancelled,2022-03-15,TD Ballpark,<NA>,NaN,<NA>,...,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
9,707019,2022,S,Spring Training,Final,2022-03-17,JetBlue Park,85,Sunny,6,...,gameInfo.firstPitch,191,<NA>,8303,Brian Knight,Jeff Nelson,Andy Fletcher,Clint Vondrak,NaN,NaN


# Validation

Completed regular-season games, and the distribution of reported conditions across all games.

In [26]:
# Show every column while inspecting; the defaults are restored at the end of the cell
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

regular_season_df = clean_df[(clean_df["game_type"] == "Regular Season") & (clean_df["game_status"] == "Final")].copy()
print("Regular season games:", len(regular_season_df))
print("Columns:", regular_season_df.shape[1])
display(regular_season_df.head(10))
print(clean_df["weather_condition"].value_counts(dropna=False))

pd.reset_option("display.max_columns")
pd.reset_option("display.width")
pd.reset_option("display.max_colwidth")

Regular season games: 12125
Columns: 25


,game_pk,season,game_type_code,game_type,game_status,game_date,venue,temperature_f,weather_condition,wind_speed_mph,wind_direction,scheduled_start_et,scheduled_start_et_minutes,first_pitch_et,first_pitch_et_minutes,first_pitch_source,game_duration_minutes,delay_duration_minutes,attendance,umpire_hp,umpire_1b,umpire_2b,umpire_3b,umpire_lf,umpire_rf
281,663178,2022,R,Regular Season,Final,2022-04-07,Wrigley Field,44,Cloudy,14,Out To CF,2022-04-07T14:20:00-04:00,860,2022-04-07T14:22:00-04:00,862,gameInfo.firstPitch,198,<NA>,35112,Ted Barrett,Lance Barksdale,Nic Lentz,Nestor Ceja,NaN,NaN
282,662766,2022,R,Regular Season,Final,2022-04-07,Kauffman Stadium,47,Cloudy,16,L To R,2022-04-07T16:10:00-04:00,970,2022-04-07T16:10:00-04:00,970,gameInfo.firstPitch,169,<NA>,28459,Phil Cuzzi,Cory Blaser,Mark Ripperger,Erich Bacchus,NaN,NaN
283,662021,2022,R,Regular Season,Final,2022-04-07,Busch Stadium,50,Cloudy,20,Out To RF,2022-04-07T16:15:00-04:00,975,2022-04-07T16:24:00-04:00,984,gameInfo.firstPitch,188,<NA>,46256,Jim Reynolds,Todd Tichenor,Mike Muchlinski,Dan Merzel,NaN,NaN
284,662571,2022,R,Regular Season,Final,2022-04-07,Nationals Park,52,Cloudy,4,Out To RF,2022-04-07T19:05:00-04:00,1145,2022-04-07T20:21:00-04:00,1221,gameInfo.firstPitch,211,76,35052,Mark Carlson,Chris Guccione,Jordan Baker,Ryan Additon,NaN,NaN
285,661577,2022,R,Regular Season,Final,2022-04-07,Truist Park,63,Partly Cloudy,15,R To L,2022-04-07T20:08:00-04:00,1208,2022-04-07T20:08:00-04:00,1208,gameInfo.firstPitch,181,<NA>,40545,Jerry Layne,Hunter Wendelstedt,Chad Whitson,Adam Hamari,NaN,NaN
286,661042,2022,R,Regular Season,Final,2022-04-07,Angel Stadium,93,Clear,6,Out To RF,2022-04-07T21:38:00-04:00,1298,2022-04-07T22:05:00-04:00,1325,gameInfo.firstPitch,195,<NA>,44723,Ed Hickox,Vic Carapazza,Jansen Visconti,Ryan Wills,NaN,NaN
287,663418,2022,R,Regular Season,Final,2022-04-07,Chase Field,93,Clear,6,L To R,2022-04-07T21:40:00-04:00,1300,2022-04-07T21:40:00-04:00,1300,gameInfo.firstPitch,198,<NA>,35508,Larry Vanover,Dan Bellino,Sean Barber,Junior Valentine,NaN,NaN
288,661333,2022,R,Regular Season,Final,2022-04-08,Yankee Stadium,59,Partly Cloudy,10,Out To CF,2022-04-08T13:05:00-04:00,785,2022-04-08T13:12:00-04:00,792,gameInfo.firstPitch,236,<NA>,46097,Laz Diaz,Brian O'Nora,Tripp Gibson,Chad Fairchild,NaN,NaN
289,661750,2022,R,Regular Season,Final,2022-04-08,Target Field,45,Sunny,17,Out To RF,2022-04-08T16:10:00-04:00,970,2022-04-08T16:16:00-04:00,976,gameInfo.firstPitch,173,<NA>,35462,Ron Kulpa,Marty Foster,Chris Conroy,Carlos Torres,NaN,NaN
290,662888,2022,R,Regular Season,Final,2022-04-08,Comerica Park,45,Cloudy,10,R To L,2022-04-08T13:10:00-04:00,790,2022-04-08T13:12:00-04:00,792,gameInfo.firstPitch,210,<NA>,43480,Marvin Hudson,Adrian Johnson,John Tumpane,Ryan Blakney,NaN,NaN


weather_condition
Partly Cloudy    4480
Clear            2622
Sunny            2322
Cloudy           1978
Roof Closed      1873
Overcast          741
Dome              392
Drizzle            67
Rain               61
NaN                57
Unknown             1
Snow                1
Name: count, dtype: int64
